# 11 — Export FreeSurfer Structural Features to CSV

This notebook turns the raw FreeSurfer `.stats` text files into four clean, raw (unnormalized) CSV tables in `results/freesurfer_features/`.

## How These Files Map to Our Longitudinal Graph $G_t = (V, E_t, X_t, u_t)$

For each participant $p$, we build a chronological sequence of brain graphs $\{G_{p,1}, \dots, G_{p,T_p}\}$, where each visit $t$ is $G_t = (V, E_t, X_t, u_t)$:
- **Edges ($E_t$):** Functional connectivity matrices already extracted in `extracted_fc_matrices/<atlas>/`.
- **Visit global vector & QC ($u_t$) + labels ($y_t$):** `freesurfer_session_global.csv` (`3,215` rows — one row per visit, sorted chronologically by `(BIDS_Subject, BIDS_Session)`).
- **Subcortical node features ($X_t^{\text{sub}}$):** `freesurfer_subcortical_14.csv` (`3,215 × 14 = 45,010` rows — shared identically by both atlases, with zero `NaN`s).
- **Cortical node features ($X_t^{\text{cort}}$):**
  - `freesurfer_cortical_schaefer200.csv` (`3,215 × 200 = 643,000` rows, zero `NaN`s)
  - `freesurfer_cortical_destrieux148.csv` (`3,215 × 148 = 475,820` rows, zero `NaN`s)

**Design choices:**
1. **Separate cortical and subcortical tables:** Cortical surface metrics (`ThickAvg`, `SurfArea`, curvature, `w-g.pct`) and subcortical volumetric metrics (`NVoxels`, `Volume_mm3`, `normMean`, `normStdDev`) live in separate tables so neither file has `NaN` padding.
2. **Raw values only:** All volumes, areas, and thicknesses are saved in raw FreeSurfer units (`mm^3`, `mm^2`, `mm`) so `eTIV` normalization and feature scaling can be done cleanly inside each cross-validation fold.
3. **No redundant coordinates:** Static `mni_x`, `mni_y`, `mni_z` coordinates stay in `data/atlases/<atlas>_roi_metadata.tsv`.
4. **Completed sessions only:** The 3 sessions that failed/lacked FreeSurfer `recon-all` (`3,218 → 3,215`) are dropped, matching `extracted_fc_matrices/`.

## 1. Setup Paths and Filter to Completed Sessions

We load `data/fmriprep-ready/master_data_file.csv`, keep the demographic, clinical, visit, and scanner columns needed for $u_t$ and targets $y_t$, drop the 3 sessions missing FreeSurfer outputs, and sort chronologically by `(BIDS_Subject, BIDS_Session)`.

In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve repository root whether notebook runs from repo root or src/gnn_connectivity_matrix/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "extracted_freesurfer_stats").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

MASTER_PATH = PROJECT_ROOT / "data" / "fmriprep-ready" / "master_data_file.csv"
FS_DIR = PROJECT_ROOT / "extracted_freesurfer_stats"
SCHAEFER_STATS_DIR = FS_DIR / "schaefer200_subcort14" / "stats"
DESTRIEUX_WG_DIR = FS_DIR / "destrieux148_subcort14" / "stats"
ATLAS_DIR = PROJECT_ROOT / "data" / "atlases"
FC_DIR = PROJECT_ROOT / "extracted_fc_matrices"
OUT_DIR = PROJECT_ROOT / "results" / "freesurfer_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCHAEFER_STEM = "Schaefer2018_200Parcels_7Networks_order"

CLINICAL_COLS = [
    "session_tag",
    "BIDS_Subject",
    "BIDS_Session",
    "Subject",
    "Diagnosis",
    "Sex",
    "Age",
    "Visit",
    "VISCODE2",
    "fMRI_Acq_Date",
    "T1_Study_Date",
    "Days_Apart",
    "MRIProtocolPhase",
    "ScannerManufacturer",
    "ScannerModel",
    "MagneticFieldStrength",
]

master_df = pd.read_csv(MASTER_PATH)
master_df["session_tag"] = master_df["BIDS_Subject"] + "_" + master_df["BIDS_Session"]
master_df = master_df[CLINICAL_COLS].copy()


def has_all_freesurfer_files(tag: str) -> bool:
    required = [
        FS_DIR / tag / "stats" / "aseg.stats",
        FS_DIR / tag / "stats" / "lh.aparc.a2009s.stats",
        FS_DIR / tag / "stats" / "rh.aparc.a2009s.stats",
        SCHAEFER_STATS_DIR / tag / f"lh.{SCHAEFER_STEM}.stats",
        SCHAEFER_STATS_DIR / tag / f"rh.{SCHAEFER_STEM}.stats",
        SCHAEFER_STATS_DIR / tag / f"lh.{SCHAEFER_STEM}.w-g.pct.stats",
        SCHAEFER_STATS_DIR / tag / f"rh.{SCHAEFER_STEM}.w-g.pct.stats",
        DESTRIEUX_WG_DIR / tag / "lh.aparc.a2009s.w-g.pct.stats",
        DESTRIEUX_WG_DIR / tag / "rh.aparc.a2009s.w-g.pct.stats",
    ]
    return all(p.is_file() and p.stat().st_size > 0 for p in required)


keep_mask = master_df["session_tag"].map(has_all_freesurfer_files)
dropped_tags = sorted(master_df.loc[~keep_mask, "session_tag"].tolist())
sessions_df = (
    master_df.loc[keep_mask]
    .sort_values(["BIDS_Subject", "BIDS_Session"])
    .reset_index(drop=True)
)

print(f"Master manifest rows:   {len(master_df):,}")
print(f"Dropped missing/failed: {len(dropped_tags)} -> {dropped_tags}")
print(f"Kept sessions:          {len(sessions_df):,} across {sessions_df['BIDS_Subject'].nunique():,} participants")
print(f"Output directory:       {OUT_DIR}")

Master manifest rows:   3,218
Dropped missing/failed: 3 -> ['sub-057S6746_ses-20190611', 'sub-127S5132_ses-20210514', 'sub-127S6147_ses-20180213']
Kept sessions:          3,215 across 1,321 participants
Output directory:       D:\Mahdi\OU\Alzheimer's Disease\Projects\gnn-connectivity-matrix\results\freesurfer_features


## 2. Load Atlas ROI Metadata (Fixed Node Order $V$)

We read the ROI metadata tables generated alongside the FC matrices (`data/atlases/*_roi_metadata.tsv`) so every ROI row order matches `extracted_fc_matrices/` 1-to-1. We also confirm that the 14 subcortical nuclei are identical across both atlases.

In [2]:
schaefer_full_meta = pd.read_csv(ATLAS_DIR / "schaefer200_subcort14_roi_metadata.tsv", sep="\t")
destrieux_full_meta = pd.read_csv(ATLAS_DIR / "destrieux148_subcort14_roi_metadata.tsv", sep="\t")

schaefer_cort_meta = schaefer_full_meta[schaefer_full_meta["network"] != "Subcortical"].reset_index(drop=True)
schaefer_cort_meta["fs_struct_name"] = schaefer_cort_meta["roi_name"]

destrieux_cort_meta = destrieux_full_meta[destrieux_full_meta["network"] != "Subcortical"].reset_index(drop=True)

subcort_meta = schaefer_full_meta[schaefer_full_meta["network"] == "Subcortical"].reset_index(drop=True)
subcort_meta["subcort_index"] = np.arange(1, len(subcort_meta) + 1)

destrieux_subcort = destrieux_full_meta[destrieux_full_meta["network"] == "Subcortical"].reset_index(drop=True)
assert subcort_meta["roi_name"].tolist() == destrieux_subcort["roi_name"].tolist()
assert subcort_meta["aseg_struct_name"].tolist() == destrieux_subcort["aseg_struct_name"].tolist()

print(f"Schaefer cortical ROIs:  {len(schaefer_cort_meta)}")
print(f"Destrieux cortical ROIs: {len(destrieux_cort_meta)}")
print(f"Shared subcortical ROIs: {len(subcort_meta)}")

Schaefer cortical ROIs:  200
Destrieux cortical ROIs: 148
Shared subcortical ROIs: 14


## 3. Simple Parsers for FreeSurfer `.stats` Files

Three lightweight functions read the standard FreeSurfer tables directly:
- `parse_aseg_stats(path)` → whole-brain header measures (`eTIV`, `BrainSegVol`, `lhSurfaceHoles`, etc.) and per-structure segmentation rows (`NVoxels`, `Volume_mm3`, intensity stats).
- `parse_aparc_stats(path)` → hemisphere header measures (`WhiteSurfArea`, `MeanThickness`) and per-parcel cortical morphometry (`NumVert`, `SurfArea`, `GrayVol`, `ThickAvg`, `ThickStd`, `MeanCurv`, `GausCurv`, `FoldInd`, `CurvInd`).
- `parse_wg_pct_stats(path)` → per-parcel gray–white matter intensity contrast (`WGPctMean`, `WGPctStd`, `WGPctSNR`).

In [3]:
IGNORED_PARCELS = {
    "unknown",
    "medial_wall",
    "background",
    "background+freesurfer_defined_medial_wall",
    "corpuscallosum",
}


def safe_float(token: str) -> float:
    try:
        val = float(token)
        return val if math.isfinite(val) else np.nan
    except (TypeError, ValueError):
        return np.nan


def parse_aseg_stats(path: Path) -> tuple[dict[str, float], dict[str, tuple]] :
    measures: dict[str, float] = {}
    structs: dict[str, tuple] = {}
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith("# Measure "):
                parts = [p.strip() for p in line[len("# Measure "):].split(",")]
                if len(parts) >= 4:
                    val = safe_float(parts[3])
                    measures[parts[0]] = val
                    measures[parts[1]] = val
                continue
            if line.startswith("#"):
                continue
            cols = line.split()
            if len(cols) >= 10:
                structs[cols[4]] = (
                    int(cols[2]),         # NVoxels
                    safe_float(cols[3]),  # Volume_mm3
                    safe_float(cols[5]),  # normMean
                    safe_float(cols[6]),  # normStdDev
                    safe_float(cols[7]),  # normMin
                    safe_float(cols[8]),  # normMax
                    safe_float(cols[9]),  # normRange
                )
    return measures, structs


def parse_aparc_stats(path: Path) -> tuple[dict[str, float], dict[str, tuple]]:
    measures: dict[str, float] = {}
    parcels: dict[str, tuple] = {}
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith("# Measure "):
                parts = [p.strip() for p in line[len("# Measure "):].split(",")]
                if len(parts) >= 4:
                    measures[parts[1]] = safe_float(parts[3])
                continue
            if line.startswith("#"):
                continue
            cols = line.split()
            if len(cols) >= 10 and cols[0].lower() not in IGNORED_PARCELS:
                parcels[cols[0]] = (
                    int(cols[1]),         # NumVert
                    safe_float(cols[2]),  # SurfArea
                    safe_float(cols[3]),  # GrayVol
                    safe_float(cols[4]),  # ThickAvg
                    safe_float(cols[5]),  # ThickStd
                    safe_float(cols[6]),  # MeanCurv
                    safe_float(cols[7]),  # GausCurv
                    int(cols[8]),         # FoldInd
                    safe_float(cols[9]),  # CurvInd
                )
    return measures, parcels


def parse_wg_pct_stats(path: Path) -> dict[str, tuple]:
    parcels: dict[str, tuple] = {}
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            cols = line.split()
            if len(cols) >= 11 and cols[4].lower() not in IGNORED_PARCELS:
                mean_val = safe_float(cols[5])
                std_val = safe_float(cols[6])
                snr_val = safe_float(cols[10])
                if math.isnan(snr_val) and std_val == 0.0:
                    snr_val = 0.0
                parcels[cols[4]] = (mean_val, std_val, snr_val)
    return parcels

## 4. Build `freesurfer_session_global.csv` ($u_t$ + QC + Clinical Targets)

One row per visit (`3,215` rows) combining:
1. **Clinical, demographic & scanner metadata** from `master_data_file.csv` (`Diagnosis`, `Sex`, `Age`, `Visit`, dates, scanner).
2. **FreeSurfer surface QC:** `lh_SurfaceHoles`, `rh_SurfaceHoles`, `total_SurfaceHoles`, and Euler numbers (`lh`, `rh`, `mean`, combined).
3. **Whole-brain summaries:** `eTIV`, `BrainSegVol`, `VentricleChoroidVol`, `CortexVol`, `SubCortGrayVol`, `lh/rh_MeanThickness`, `lh/rh_WhiteSurfArea`, etc.
4. **Extra non-node `aseg.stats` volumes:** ventricles, `WM-hypointensities`, corpus callosum (`CC_*`), cerebellum, `Brain-Stem`, `VentralDC`, choroid plexus, optic chiasm, and vessels.

In [4]:
GLOBAL_MEASURE_COLS = [
    "eTIV",
    "BrainSegVol",
    "BrainSegVolNotVent",
    "VentricleChoroidVol",
    "CortexVol",
    "lhCortexVol",
    "rhCortexVol",
    "CerebralWhiteMatterVol",
    "lhCerebralWhiteMatterVol",
    "rhCerebralWhiteMatterVol",
    "SubCortGrayVol",
    "TotalGrayVol",
    "SupraTentorialVol",
    "SupraTentorialVolNotVent",
]

EXTRA_ASEG_STRUCTS = [
    "Left-Lateral-Ventricle",
    "Right-Lateral-Ventricle",
    "Left-Inf-Lat-Vent",
    "Right-Inf-Lat-Vent",
    "3rd-Ventricle",
    "4th-Ventricle",
    "CSF",
    "WM-hypointensities",
    "CC_Posterior",
    "CC_Mid_Posterior",
    "CC_Central",
    "CC_Mid_Anterior",
    "CC_Anterior",
    "Left-Cerebellum-Cortex",
    "Right-Cerebellum-Cortex",
    "Left-Cerebellum-White-Matter",
    "Right-Cerebellum-White-Matter",
    "Brain-Stem",
    "Left-VentralDC",
    "Right-VentralDC",
    "Left-choroid-plexus",
    "Right-choroid-plexus",
    "Optic-Chiasm",
    "Left-vessel",
    "Right-vessel",
]

global_rows = []
for tag in sessions_df["session_tag"]:
    aseg_meas, aseg_structs = parse_aseg_stats(FS_DIR / tag / "stats" / "aseg.stats")
    lh_meas, _ = parse_aparc_stats(FS_DIR / tag / "stats" / "lh.aparc.a2009s.stats")
    rh_meas, _ = parse_aparc_stats(FS_DIR / tag / "stats" / "rh.aparc.a2009s.stats")

    lh_holes = int(round(aseg_meas["lhSurfaceHoles"]))
    rh_holes = int(round(aseg_meas["rhSurfaceHoles"]))
    lh_euler = 2 - 2 * lh_holes
    rh_euler = 2 - 2 * rh_holes

    row = {
        "session_tag": tag,
        "lh_SurfaceHoles": lh_holes,
        "rh_SurfaceHoles": rh_holes,
        "total_SurfaceHoles": lh_holes + rh_holes,
        "lh_euler_number": lh_euler,
        "rh_euler_number": rh_euler,
        "mean_euler_number": 0.5 * (lh_euler + rh_euler),
        "euler_number": 2 - 2 * (lh_holes + rh_holes),
    }
    for col in GLOBAL_MEASURE_COLS:
        row[col] = aseg_meas[col]

    row["lh_MeanThickness"] = lh_meas["MeanThickness"]
    row["rh_MeanThickness"] = rh_meas["MeanThickness"]
    row["lh_WhiteSurfArea"] = lh_meas["WhiteSurfArea"]
    row["rh_WhiteSurfArea"] = rh_meas["WhiteSurfArea"]

    for s_name in EXTRA_ASEG_STRUCTS:
        row[f"{s_name}_Volume_mm3"] = aseg_structs[s_name][1]

    global_rows.append(row)

df_global_fs = pd.DataFrame(global_rows)
df_global = sessions_df.merge(df_global_fs, on="session_tag", how="inner", validate="one_to_one")

global_csv_path = OUT_DIR / "freesurfer_session_global.csv"
df_global.to_csv(global_csv_path, index=False)
print(f"Saved {global_csv_path.name}: {df_global.shape[0]:,} rows x {df_global.shape[1]} cols ({global_csv_path.stat().st_size / 1e6:.2f} MB)")
print(df_global[["session_tag", "Diagnosis", "Age", "Sex", "euler_number", "eTIV", "WM-hypointensities_Volume_mm3"]].head(3))

Saved freesurfer_session_global.csv: 3,215 rows x 66 cols (1.64 MB)
                 session_tag  ... WM-hypointensities_Volume_mm3
0  sub-002S0295_ses-20120510  ...                        3367.8
1  sub-002S0413_ses-20120515  ...                        2656.7
2  sub-002S0413_ses-20130510  ...                        2927.9

[3 rows x 7 columns]


## 5. Build `freesurfer_subcortical_14.csv` ($X_t^{\text{sub}}$ — Shared Across Atlases)

Long format (`3,215 × 14 = 45,010` rows) ordered by `(BIDS_Subject, BIDS_Session, subcort_index)`. `subcort_index` `1..14` aligns directly with nodes `201..214` in `schaefer200_subcort14` and nodes `149..162` in `destrieux148_subcort14`.

In [5]:
SUBCORT_META_LIST = [
    (int(r.subcort_index), str(r.roi_name), str(r.aseg_struct_name), str(r.hemisphere))
    for r in subcort_meta.itertuples(index=False)
]

ASEG_ALIASES = {
    "Left-Thalamus": ("Left-Thalamus", "Left-Thalamus-Proper"),
    "Right-Thalamus": ("Right-Thalamus", "Right-Thalamus-Proper"),
}

subcort_rows = []
for tag in sessions_df["session_tag"]:
    _, aseg_structs = parse_aseg_stats(FS_DIR / tag / "stats" / "aseg.stats")
    for sub_idx, roi_name, aseg_name, hemi in SUBCORT_META_LIST:
        candidates = ASEG_ALIASES.get(aseg_name, (aseg_name,))
        vals = next(aseg_structs[c] for c in candidates if c in aseg_structs)
        subcort_rows.append((tag, sub_idx, roi_name, aseg_name, hemi, *vals))

SUBCORT_COLS = [
    "session_tag",
    "subcort_index",
    "roi_name",
    "aseg_struct_name",
    "hemisphere",
    "NVoxels",
    "Volume_mm3",
    "normMean",
    "normStdDev",
    "normMin",
    "normMax",
    "normRange",
]

df_subcort = pd.DataFrame(subcort_rows, columns=SUBCORT_COLS)
subcort_csv_path = OUT_DIR / "freesurfer_subcortical_14.csv"
df_subcort.to_csv(subcort_csv_path, index=False)
print(f"Saved {subcort_csv_path.name}: {df_subcort.shape[0]:,} rows x {df_subcort.shape[1]} cols ({subcort_csv_path.stat().st_size / 1e6:.2f} MB)")
print(f"Null entries: {int(df_subcort.isna().sum().sum())}")
print(df_subcort.head(3))

Saved freesurfer_subcortical_14.csv: 45,010 rows x 12 cols (5.12 MB)
Null entries: 0
                 session_tag  subcort_index  ... normMax normRange
0  sub-002S0295_ses-20120510              1  ...   108.0      75.0
1  sub-002S0295_ses-20120510              2  ...   106.0      64.0
2  sub-002S0295_ses-20120510              3  ...   104.0      48.0

[3 rows x 12 columns]


## 6. Build `freesurfer_cortical_schaefer200.csv` ($X_t^{\text{cort}}$ for Schaefer 200)

Long format (`3,215 × 200 = 643,000` rows) ordered by `(BIDS_Subject, BIDS_Session, roi_index)`. Combines the 9 standard surface morphometry columns from `{lh,rh}.Schaefer2018_200Parcels_7Networks_order.stats` with the 3 gray–white contrast metrics from `.w-g.pct.stats`. Medial-wall-clipped parcels with `WGPctStd == 0.0` produce `-nan` (`0/0`) for `WGPctSNR` in FreeSurfer `mri_segstats` and are set to `0.0`.

In [6]:
SCHAEFER_META_LIST = [
    (int(r.roi_index), str(r.roi_name), str(r.hemisphere), str(r.network), str(r.fs_struct_name))
    for r in schaefer_cort_meta.itertuples(index=False)
]

schaefer_rows = []
for tag in sessions_df["session_tag"]:
    _, lh_parcels = parse_aparc_stats(SCHAEFER_STATS_DIR / tag / f"lh.{SCHAEFER_STEM}.stats")
    _, rh_parcels = parse_aparc_stats(SCHAEFER_STATS_DIR / tag / f"rh.{SCHAEFER_STEM}.stats")
    lh_wg = parse_wg_pct_stats(SCHAEFER_STATS_DIR / tag / f"lh.{SCHAEFER_STEM}.w-g.pct.stats")
    rh_wg = parse_wg_pct_stats(SCHAEFER_STATS_DIR / tag / f"rh.{SCHAEFER_STEM}.w-g.pct.stats")

    for roi_idx, roi_name, hemi, network, fs_name in SCHAEFER_META_LIST:
        aparc_vals = lh_parcels[fs_name] if hemi == "LH" else rh_parcels[fs_name]
        wg_vals = lh_wg[fs_name] if hemi == "LH" else rh_wg[fs_name]
        schaefer_rows.append((tag, roi_idx, roi_name, hemi, network, *aparc_vals, *wg_vals))

SCHAEFER_COLS = [
    "session_tag",
    "roi_index",
    "roi_name",
    "hemisphere",
    "network",
    "NumVert",
    "SurfArea",
    "GrayVol",
    "ThickAvg",
    "ThickStd",
    "MeanCurv",
    "GausCurv",
    "FoldInd",
    "CurvInd",
    "WGPctMean",
    "WGPctStd",
    "WGPctSNR",
]

df_schaefer = pd.DataFrame(schaefer_rows, columns=SCHAEFER_COLS)
schaefer_csv_path = OUT_DIR / "freesurfer_cortical_schaefer200.csv"
df_schaefer.to_csv(schaefer_csv_path, index=False)
print(f"Saved {schaefer_csv_path.name}: {df_schaefer.shape[0]:,} rows x {df_schaefer.shape[1]} cols ({schaefer_csv_path.stat().st_size / 1e6:.2f} MB)")
print(f"Zero-contrast parcels (WGPctStd == 0.0): {int((df_schaefer['WGPctStd'] == 0.0).sum())}")
print(f"Null entries: {int(df_schaefer.isna().sum().sum())}")
print(df_schaefer.head(3))

Saved freesurfer_cortical_schaefer200.csv: 643,000 rows x 17 cols (87.89 MB)
Zero-contrast parcels (WGPctStd == 0.0): 32
Null entries: 0
                 session_tag  roi_index  ... WGPctStd WGPctSNR
0  sub-002S0295_ses-20120510          1  ...   6.2115   3.2786
1  sub-002S0295_ses-20120510          2  ...   5.3591   4.0623
2  sub-002S0295_ses-20120510          3  ...   5.8884   4.1925

[3 rows x 17 columns]


## 7. Build `freesurfer_cortical_destrieux148.csv` ($X_t^{\text{cort}}$ for Destrieux 148)

Long format (`3,215 × 148 = 475,820` rows) ordered by `(BIDS_Subject, BIDS_Session, roi_index)` from FreeSurfer's `{lh,rh}.aparc.a2009s.stats` plus the 3 gray–white contrast metrics from `extracted_freesurfer_stats/destrieux148_subcort14/stats/` and the `network` column (`"Cortical"`) so both cortical CSVs share the exact same 17-column schema. FreeSurfer `mris_anatomical_stats` omits a small anatomically variable parcel (e.g., `S_interm_prim-Jensen`) when 0 vertices survive `cortex.label`; omitted 0-vertex parcels are recorded as zeros (`NumVert = 0`) so every session keeps the full 148-row grid.

In [7]:
DESTRIEUX_META_LIST = [
    (int(r.roi_index), str(r.roi_name), str(r.hemisphere), str(r.network), str(r.fs_struct_name))
    for r in destrieux_cort_meta.itertuples(index=False)
]
EMPTY_APARC_ROW = (0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0, 0.0)
EMPTY_WG_ROW = (0.0, 0.0, 0.0)

destrieux_rows = []
n_zero_vert_omitted = 0
affected_sessions = set()
for tag in sessions_df["session_tag"]:
    _, lh_parcels = parse_aparc_stats(FS_DIR / tag / "stats" / "lh.aparc.a2009s.stats")
    _, rh_parcels = parse_aparc_stats(FS_DIR / tag / "stats" / "rh.aparc.a2009s.stats")
    lh_wg = parse_wg_pct_stats(DESTRIEUX_WG_DIR / tag / "lh.aparc.a2009s.w-g.pct.stats")
    rh_wg = parse_wg_pct_stats(DESTRIEUX_WG_DIR / tag / "rh.aparc.a2009s.w-g.pct.stats")

    for roi_idx, roi_name, hemi, network, fs_name in DESTRIEUX_META_LIST:
        parcel_map = lh_parcels if hemi == "LH" else rh_parcels
        wg_map = lh_wg if hemi == "LH" else rh_wg
        aparc_vals = parcel_map.get(fs_name)
        if aparc_vals is None:
            aparc_vals = EMPTY_APARC_ROW
            n_zero_vert_omitted += 1
            affected_sessions.add(tag)
        wg_vals = wg_map.get(fs_name, EMPTY_WG_ROW)
        destrieux_rows.append((tag, roi_idx, roi_name, hemi, network, *aparc_vals, *wg_vals))

DESTRIEUX_COLS = SCHAEFER_COLS

df_destrieux = pd.DataFrame(destrieux_rows, columns=DESTRIEUX_COLS)
destrieux_csv_path = OUT_DIR / "freesurfer_cortical_destrieux148.csv"
df_destrieux.to_csv(destrieux_csv_path, index=False)
print(f"Saved {destrieux_csv_path.name}: {df_destrieux.shape[0]:,} rows x {df_destrieux.shape[1]} cols ({destrieux_csv_path.stat().st_size / 1e6:.2f} MB)")
print(f"Omitted 0-vertex parcels filled with zeros: {n_zero_vert_omitted} across {len(affected_sessions)} sessions")
print(f"Zero-contrast parcels (WGPctStd == 0.0): {int((df_destrieux['WGPctStd'] == 0.0).sum())}")
print(f"Null entries: {int(df_destrieux.isna().sum().sum())}")
print(df_destrieux.head(3))

Saved freesurfer_cortical_destrieux148.csv: 475,820 rows x 17 cols (63.50 MB)
Omitted 0-vertex parcels filled with zeros: 25 across 17 sessions
Zero-contrast parcels (WGPctStd == 0.0): 25
Null entries: 0
                 session_tag  roi_index  ... WGPctStd WGPctSNR
0  sub-002S0295_ses-20120510          1  ...   6.4132   4.4190
1  sub-002S0295_ses-20120510          2  ...   5.5877   4.0124
2  sub-002S0295_ses-20120510          3  ...   4.9102   3.0803

[3 rows x 17 columns]


## 8. Verify Data Integrity & 1-to-1 Alignment with FC Matrices

Final checks before modeling:
1. Exact row counts (`3,215` visits, `14` subcortical, `200` Schaefer, `148` Destrieux).
2. Zero `NaN`s in all three node feature tables and in all FreeSurfer + `Diagnosis`/`Age`/`Sex` columns of `freesurfer_session_global.csv`.
3. Exact session match against `extracted_fc_matrices/schaefer200_subcort14/` and `extracted_fc_matrices/destrieux148_subcort14/`.
4. Exact `[cortical] + [subcortical]` node ordering match against the FC matrix dimensions (`214 × 214` and `162 × 162`).

In [8]:
n_sessions = len(df_global)
assert n_sessions == 3215, f"Expected 3215 sessions, got {n_sessions}"
assert len(df_subcort) == n_sessions * 14
assert len(df_schaefer) == n_sessions * 200
assert len(df_destrieux) == n_sessions * 148
assert df_destrieux.columns.tolist() == df_schaefer.columns.tolist()

# Zero NaNs in all node tables, and zero NaNs in FreeSurfer + core clinical columns
assert df_subcort.isna().sum().sum() == 0
assert df_schaefer.isna().sum().sum() == 0
assert df_destrieux.isna().sum().sum() == 0
fs_and_core_cols = [c for c in df_global.columns if c not in {"VISCODE2", "MRIProtocolPhase", "ScannerManufacturer", "ScannerModel", "MagneticFieldStrength"}]
assert df_global[fs_and_core_cols].isna().sum().sum() == 0

# Identical chronological session order across all 4 tables
expected_tags = df_global["session_tag"].tolist()
assert df_subcort["session_tag"].iloc[::14].tolist() == expected_tags
assert df_schaefer["session_tag"].iloc[::200].tolist() == expected_tags
assert df_destrieux["session_tag"].iloc[::148].tolist() == expected_tags

# Exact match with FC matrices on disk
for atlas_name, cort_df, expected_dim in [
    ("schaefer200_subcort14", df_schaefer, 214),
    ("destrieux148_subcort14", df_destrieux, 162),
]:
    fc_atlas_dir = FC_DIR / atlas_name
    if fc_atlas_dir.is_dir():
        fc_tags = {p.stem for p in fc_atlas_dir.glob("sub-*.npz")}
        assert set(expected_tags) == fc_tags, f"Session mismatch for {atlas_name}"

        sample_fc = np.load(fc_atlas_dir / f"{expected_tags[0]}.npz")
        assert sample_fc["pearson_fc"].shape == (expected_dim, expected_dim)

        fc_meta = pd.read_csv(fc_atlas_dir / f"atlas_{atlas_name}_roi_metadata.tsv", sep="\t")
        n_cort = expected_dim - 14
        combined_names = (
            cort_df.iloc[:n_cort]["roi_name"].tolist()
            + df_subcort.iloc[:14]["roi_name"].tolist()
        )
        assert combined_names == fc_meta["roi_name"].tolist()

summary = []
for path, df in [
    (global_csv_path, df_global),
    (subcort_csv_path, df_subcort),
    (schaefer_csv_path, df_schaefer),
    (destrieux_csv_path, df_destrieux),
]:
    summary.append({
        "file": path.name,
        "rows": len(df),
        "cols": df.shape[1],
        "size_mb": round(path.stat().st_size / 1e6, 2),
        "total_nans": int(df.isna().sum().sum()),
    })

print("ALL CHECKS PASSED!")
print(pd.DataFrame(summary).to_string(index=False))

ALL CHECKS PASSED!
                                file   rows  cols  size_mb  total_nans
       freesurfer_session_global.csv   3215    66     1.64          14
       freesurfer_subcortical_14.csv  45010    12     5.12           0
 freesurfer_cortical_schaefer200.csv 643000    17    87.89           0
freesurfer_cortical_destrieux148.csv 475820    17    63.50           0
